In [28]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
import os
import sys
import sqlite3
import seaborn.objects as so


In [29]:
def sqlite_to_dfs(sqlite_file):
    """
    Read all tables and views from a SQLite file and return a dict mapping
    table/view name -> pandas.DataFrame.

    Parameters
    ----------
    sqlite_file : str or os.PathLike
        Path to the .sqlite file.

    Returns
    -------
    dict
        {table_name: DataFrame, ...}
    """
    conn = sqlite3.connect(sqlite_file)
    try:
        cur = conn.cursor()
        cur.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%';"
        )
        tables = [row[0] for row in cur.fetchall()]
        dfs = {}
        for tbl in tables:
            # quote table name to handle odd characters
            query = f"SELECT * FROM \"{tbl}\""
            dfs[tbl] = pd.read_sql_query(query, conn)
        return dfs
    finally:
        conn.close()

In [6]:
# ============================================================================
# HELPER FUNCTIONS - Reusable logic for efficiency calculations
# ============================================================================

def extract_vehicle_demand_mapping(eff_data):
    """
    Extract vehicle demand mapping from efficiency data.
    Returns: DataFrame with 'demand' and 'tech' columns
    """
    df = eff_data.loc[eff_data['output_comm'].str.startswith('T_D_')].copy()
    df['demand_type'] = df['output_comm'].apply(lambda x: '_'.join(x.split('_')[:-1]))
    
    # Build mapping
    vdm_list = []
    for demand in df['demand_type'].unique():
        techs = df.loc[df['demand_type'] == demand, 'tech'].unique()
        for tech in techs:
            vdm_list.append({'demand': demand, 'tech': tech})
    
    return pd.DataFrame(vdm_list)


def calculate_virtual_demand(df_c2a, df_cf, df_ec, vdm, region):
    """
    Calculate virtual demand by merging capacity, factor, and existing capacity.
    Eliminates redundant calculations.
    
    Parameters:
    -----------
    df_c2a : DataFrame
        Capacity to Activity data
    df_cf : DataFrame
        Capacity Factor data
    df_ec : DataFrame
        Existing/Net Capacity data
    vdm : DataFrame
        Vehicle Demand Mapping (tech -> demand)
    region : str
        Region to filter by
    
    Returns:
    --------
    DataFrame with aggregated virtual demand by demand_type and vintage
    """
    # Filter and merge
    df = pd.merge(
        df_c2a[df_c2a['region'] == region],
        df_cf[df_cf['region'] == region],
        on=['tech', 'region'],
        how='inner'
    )
    df = pd.merge(df, df_ec, on=['tech', 'vintage'], how='inner')
    
    # Calculate virtual demand and map demand type
    df['virtual_demand'] = df['c2a'] * df['factor'] * df['capacity']
    df['demand_type'] = df['tech'].map(vdm.set_index('tech')['demand'])
    
    # Aggregate
    return df.groupby(['demand_type', 'vintage'], as_index=False)['virtual_demand'].sum()


def calculate_survival_factors_vectorized(df_ec, df_sc, periods):
    """
    Vectorized calculation of survival factors.
    Much more efficient than row iteration.
    
    Parameters:
    -----------
    df_ec : DataFrame
        Existing capacity data
    df_sc : DataFrame
        Survival curve data
    periods : list
        List of periods to calculate survival for
    
    Returns:
    --------
    DataFrame with survival factors applied to capacity
    """
    # Create a temp DataFrame with capacity data
    result_df = df_ec[['tech', 'vintage', 'capacity']].reset_index(drop=True)
    
    # Merge with survival curve data
    merged = pd.merge(
        result_df,
        df_sc,
        on=['tech', 'vintage'],
        how='left'
    )
    
    # For each period, calculate surviving capacity using vectorized operations
    for period in periods:
        # Filter to current period and pivot fraction values
        period_data = df_sc[df_sc['period'] == period].set_index(['tech', 'vintage'])['fraction']
        
        merged[f'factor_{period}'] = merged.set_index(['tech', 'vintage']).index.map(
            lambda x: period_data.get(x, 1.0)
        )
        merged[f's_cap{period}'] = np.where(
            merged['vintage'] <= period,
            merged['capacity'] * merged[f'factor_{period}'],
            0
        )
    
    # Reshape and return
    survival_cols = [col for col in merged.columns if col.startswith('s_cap')]
    result = merged[['tech', 'vintage'] + survival_cols].copy()
    
    return result


def format_demand_data(existing_df, future_df):
    """
    Format and combine existing and future demand data.
    Extracts vehicle type and ensures consistent structure.
    """
    # Process existing demand
    df_ex = existing_df.copy()
    df_ex['demand_type_2'] = df_ex['demand_type'].apply(lambda x: x.split('_')[-1])
    df_ex = df_ex.groupby(['demand_type_2', 'vintage'], as_index=False)['virtual_demand'].sum()
    df_ex['demand'] = df_ex['virtual_demand']
    df_ex = df_ex[['demand_type_2', 'vintage', 'demand']]
    
    # Process future demand
    df_fut = future_df.copy()
    df_fut['demand_type_2'] = df_fut['demand_type'].apply(lambda x: x.split('_')[-1])
    df_fut = df_fut.groupby(['demand_type_2', 'period'], as_index=False)['demand'].sum()
    df_fut['vintage'] = df_fut['period']
    df_fut = df_fut[['demand_type_2', 'vintage', 'demand']]
    
    # Combine
    combined = pd.concat([df_ex, df_fut], ignore_index=True)
    combined.rename(columns={'demand_type_2': 'Vehicle Type'}, inplace=True)
    
    return combined


In [27]:
output_file = 'canoe_hr_16d.sqlite'
output_dir = '../outputs/canoe_report/'+output_file

data = sqlite_to_dfs(output_dir)

In [8]:
ec = data['ExistingCapacity']
demand = data['Demand']
o_bc = data['OutputBuiltCapacity']


KeyError: 'ExistingCapacity'

In [9]:
ec

NameError: name 'ec' is not defined

In [10]:
df = data['Technology']
df_transport = df.loc[df['sector']=='transportation']
df_transport.to_csv("../dbs/vehicle_classification.csv", index=False)
df = df_transport


KeyError: 'Technology'

In [11]:
eff = data['Efficiency']
eff = eff.loc[eff['tech'].isin(df['tech'])].copy()


KeyError: 'Efficiency'

In [12]:
eff = eff.groupby(['tech','output_comm'], as_index=False)['efficiency'].mean()
eff.to_csv("../dbs/vehicle_efficiency.csv", index=False)



NameError: name 'eff' is not defined

In [13]:
# Extract vehicle demand mapping
df_eff_transport = eff.loc[eff['output_comm'].str.startswith('T_D_')].copy()
df_eff_transport['demand_type'] = df_eff_transport['output_comm'].apply(lambda x: '_'.join(x.split('_')[:-1]))

vc = {}
for demand in df_eff_transport['demand_type'].unique():
    vc[demand] = df_eff_transport.loc[df_eff_transport['demand_type'] == demand, 'tech'].to_list()


NameError: name 'eff' is not defined

In [14]:
# Create vehicle demand mapping
vdm_list = []
for demand, techs in vc.items():
    print(f"{demand}: {techs}")
    for tech in techs:
        vdm_list.append({'demand': demand, 'tech': tech})

vdm = pd.DataFrame(vdm_list)
vdm


NameError: name 'vc' is not defined

In [15]:
import seaborn.objects as so
vc_ = {}
vc_['T_D_pkm_ldv'] = vc['T_D_pkm_ldv'] # Just to focus on one demand for now

for demand, techs in vc_.items():
    print(f"{demand}: {techs}\n")

    # Assuming 'ec' is your dataframe and you might be filtering it 
    # based on 'demand' or 'techs' inside this loop?
    df = ec.loc[ec['tech'].isin(techs)].copy() 
    
    fig = (
        so.Plot(df, x='vintage', y='capacity', color='tech')
        .add(so.Bars(), so.Agg(), so.Stack())
        .label(title=f"Existing Capacity: {demand}") # Good for keeping track in a loop
    )
    
    # Use the object's show method
    fig.show()

NameError: name 'vc' is not defined

In [16]:
vdm

NameError: name 'vdm' is not defined

In [17]:
# Filter data to transportation sector technologies
c2a = data['CapacityToActivity']
c2a = c2a.loc[c2a['tech'].isin(vdm['tech'])]

cf = data['LimitAnnualCapacityFactor']
cf = cf.loc[cf['tech'].isin(vdm['tech'])]

ec = ec.loc[ec['tech'].isin(vdm['tech'])]


KeyError: 'CapacityToActivity'

In [18]:
# Calculate existing virtual demand
region = 'ON'

# Filter EC data by region
df_ec_region = ec.loc[ec['region'] == region]

# Calculate virtual demand using helper function
existing = calculate_virtual_demand(c2a, cf, df_ec_region, vdm, region)


NameError: name 'ec' is not defined

In [19]:
dm = data['Demand'].copy()
dm = dm.loc[dm['region'] == 'ON']
dm = dm.loc[dm['commodity'].str.startswith('T_D_')]
dm['demand_type'] = dm['commodity'].apply(lambda x: '_'.join(x.split('_')[:-1]))
future = dm.groupby(['demand_type', 'period'], as_index=False)['demand'].sum()


KeyError: 'Demand'

In [20]:
existing

NameError: name 'existing' is not defined

In [21]:
df_f = future.copy()
df_f['demand_type_2'] = df_f['demand_type'].apply(lambda x: x.split('_')[-1]) 
df_f = df_f.groupby(['demand_type_2', 'period'], as_index=False).sum()
df_f.drop(columns=['demand_type'], inplace=True)
df_f['vintage'] = df_f['period']


NameError: name 'future' is not defined

In [22]:
# Format existing demand data
df_ex = existing.copy()
df_ex['demand_type_2'] = df_ex['demand_type'].apply(lambda x: x.split('_')[-1])
df_ex = df_ex.groupby(['demand_type_2', 'vintage'], as_index=False)['virtual_demand'].sum()
df_ex['demand'] = df_ex['virtual_demand']
df_ex = df_ex[['demand_type_2', 'vintage', 'demand']]


NameError: name 'existing' is not defined

In [23]:
# Combine and visualize demand data
df = pd.concat([df_ex.loc[:,['demand_type_2','vintage','demand']], df_f.loc[:,['demand_type_2','vintage','demand']]], ignore_index=True)

df.rename(columns={'demand_type_2': 'Vehicle Type'}, inplace=True)
fig = so.Plot(df, x='vintage', y='demand', color='Vehicle Type').add(so.Line()).label(title="Total Demand (Existing + Future)")
fig.show()


NameError: name 'df_ex' is not defined

In [24]:
# Load survival curve data
sc = data['LifetimeSurvivalCurve']
periods = [2000, 2005, 2010, 2015, 2020]


KeyError: 'LifetimeSurvivalCurve'

In [25]:
# Filter existing capacity for ON region
df_ec = ec.loc[ec['region']=='ON'][['tech', 'vintage', 'capacity']].copy()

# Filter survival curve for ON region
df_sc = sc.loc[sc['region']=='ON'].copy()

# Merge survival curve with existing capacity
df_merged = pd.merge(df_ec, df_sc, on=['tech', 'vintage'], how='left')

# Calculate surviving capacity for each period
for period in periods:
    df_merged[f's_cap{period}'] = np.where(
        df_merged['vintage'] <= period,
        df_merged['capacity'] * df_merged.loc[df_merged['period'] == period, 'fraction'].fillna(1),
        0
    )

# Extract surviving capacity columns and prepare for next steps
df_net = df_merged[['tech', 'vintage'] + [f's_cap{p}' for p in periods]].copy()
df_net


NameError: name 'ec' is not defined

In [26]:
# ============================================================================
# FINAL VIRTUAL DEMAND CALCULATION WITH SURVIVAL CURVES
# ============================================================================

region = 'ON'

# Calculate virtual demand using helper function with net capacity
df_virtual = calculate_virtual_demand(c2a, cf, df_net, vdm, region)

# Format and export results
df_virtual.to_csv("../dbs/virtual_demand.csv", index=False)

# Combine existing and future demand using the helper function
combined_demand = format_demand_data(df_virtual, future)

# Visualize combined demand
fig = (
    so.Plot(combined_demand, x='vintage', y='demand', color='Vehicle Type')
    .add(so.Line())
    .label(title="Total Demand (Existing + Future)")
)
fig.show()


NameError: name 'c2a' is not defined